# Perron: Cross-Architecture Scaling Surface & Generality Benchmark
### Protocol Reference: PREREG-SCALING-v1 (Held-Out SWE-bench Verified Stratum C)

**Author**: Sultan Haikal (GitHub: `@yvliet`)  
**Target**: Track 2 (Gemma 4 Developer Agent Paper Track)  
**Hardware**: Dual NVIDIA Tesla T4 GPUs (2x 15GB GDDR6)  
**License**: Apache-2.0

---

## 1. Executive Protocol & Hypotheses

Evaluates candidate discrimination (ROC-AUC) across a 5-tier 2025–2026 open-weight model gradient spanning:
1. **`google/gemma-4-e2b-it`** (2.0B Dense PLE, April 2026)
2. **`google/gemma-4-e4b-it`** (4.0B Dense PLE, April 2026)
3. **`microsoft/Phi-4-mini-instruct`** (3.8B Dense Reasoning SLM, Feb 2025)
4. **`deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`** (7.0B Reasoning Distill, Jan 2025)
5. **`google/gemma-4-26b-a4b-it`** (3.8B active / 26B total MoE, April 2026)

Tests whether the **Lexical Abstraction Bottleneck** (AUC <= 0.6596) is scale-invariant zero-shot ($H_1$) and whether dynamic MoE routing provides representational efficiency over dense baselines ($H_2$).

In [ ]:
# Step 1: Hardware Environment & CUDA Telemetry
import os, sys, psutil, time, json, gc, re
from pathlib import Path
import torch

print(f'Python Version: {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
cuda_ok = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if cuda_ok else 0
print(f'CUDA Available: {cuda_ok} (Count: {num_gpus})')

if cuda_ok:
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}]: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB')
else:
    print('Operating in CPU / Emulation mode.')


In [ ]:
# Step 2: Install dependencies if missing
!pip install -q bitsandbytes accelerate scipy transformers


In [ ]:
# Step 3: Locate Stratum C Evaluation Bundle
bundle_candidates = [
    Path('/kaggle/input/perron-gemma4-benchmark/stratum_c_eval_bundle.json'),
    Path('/kaggle/input/stratum_c_eval_bundle.json'),
    Path('data_release/stratum_c_eval_bundle.json'),
    Path('../data_release/stratum_c_eval_bundle.json'),
]

bundle_path = None
for p in bundle_candidates:
    if p.exists():
        bundle_path = p
        break

if bundle_path is None:
    base_in = Path('/kaggle/input')
    if base_in.exists():
        for p in base_in.rglob('stratum_c_eval_bundle.json'):
            if p.exists():
                bundle_path = p
                break

print(f'[+] Using evaluation bundle: {bundle_path}')
with open(bundle_path, 'r', encoding='utf-8') as f:
    bundle_data = json.load(f)
tasks = bundle_data.get('tasks', [])
print(f'[+] Loaded {len(tasks)} tasks.')


In [ ]:
# Step 4: Define Evaluation Worker & Scoring Functions
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PROMPT_P0 = (
    "You are an expert software engineer reviewing a repository to locate a reported defect.\n\n"
    "Problem Statement:\n{issue_text}\n\n"
    "Candidate Code Element:\n{candidate_breadcrumb}\n\n"
    "Does this candidate code element contain or directly cause the reported bug? Answer 'Yes' or 'No'."
)

def truncate_issue_text(text: str, max_chars: int = 3000) -> str:
    if not text: return ''
    clean = re.sub(r'\s+', ' ', text).strip()
    if len(clean) <= max_chars: return clean
    half = (max_chars - 60) // 2
    return clean[:half] + ' ... [TRUNCATED] ... ' + clean[-half:]

def format_candidate_breadcrumb(cand: dict) -> str:
    sym_name = cand.get('symbol_name') or cand.get('identifier') or 'unknown'
    fpath = cand.get('file_path') or cand.get('file') or 'unknown'
    lines = cand.get('lines') or [cand.get('line_start', 1), cand.get('line_end', 1)]
    snippet = cand.get('code') or cand.get('snippet') or f'# {sym_name}'
    return f'File: {fpath} (Lines {lines[0]}-{lines[1]})\nSymbol: {sym_name}\n```python\n{snippet}\n```'

def resolve_yes_no_token_ids(tok):
    y_ids, n_ids = set(), set()
    for w in ['Yes', ' Yes', 'YES', ' yes']:
        enc = tok.encode(w, add_special_tokens=False)
        if enc: y_ids.add(enc[-1])
    for w in ['No', ' No', 'NO', ' no']:
        enc = tok.encode(w, add_special_tokens=False)
        if enc: n_ids.add(enc[-1])
    return sorted(list(y_ids)), sorted(list(n_ids))


In [ ]:
# Step 5: Benchmark Execution Across 5 Modern Models
MODELS_CONFIG = [
    {'id': 'google/gemma-4-e2b-it', 'name': 'Gemma 4 E2B', 'params_b': 2.0, 'active_params_b': 2.0, 'archetype': 'PLE Dense Edge', 'tier': 'pointwise'},
    {'id': 'google/gemma-4-e4b-it', 'name': 'Gemma 4 E4B', 'params_b': 4.0, 'active_params_b': 4.0, 'archetype': 'PLE Dense Edge', 'tier': 'hybrid_anchor'},
    {'id': 'microsoft/Phi-4-mini-instruct', 'name': 'Phi-4-mini', 'params_b': 3.8, 'active_params_b': 3.8, 'archetype': 'High-Density Synthetic Reasoning', 'tier': 'pointwise'},
    {'id': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', 'name': 'DeepSeek R1 Distill 7B', 'params_b': 7.0, 'active_params_b': 7.0, 'archetype': 'RL Reasoning Distill', 'tier': 'pointwise'},
    {'id': 'google/gemma-4-26b-a4b-it', 'name': 'Gemma 4 26B-A4B', 'params_b': 26.0, 'active_params_b': 3.8, 'archetype': 'Dynamic Sparse MoE', 'tier': 'hybrid_anchor'},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if torch.cuda.is_available() else None

all_model_results = []

for m_spec in MODELS_CONFIG:
    m_id = m_spec['id']
    print('=' * 80)
    print(f'Evaluating Model: {m_id} ({m_spec["archetype"]})')
    print('=' * 80)
    t_start = time.time()

    try:
        tok = AutoTokenizer.from_pretrained(m_id, trust_remote_code=True)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        tok.padding_side = 'left'

        model = AutoModelForCausalLM.from_pretrained(
            m_id,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True,
            torch_dtype=torch.float16,
        ) if torch.cuda.is_available() else None

        y_ids, n_ids = resolve_yes_no_token_ids(tok)
        print(f'Classification token IDs: Yes={y_ids}, No={n_ids}')

        task_aucs = []
        for t in tasks:
            issue = truncate_issue_text(t.get('problem_statement', ''))
            cands = t.get('candidates', [])
            gold_ids = set(t.get('gold_symbol_ids', [])) or set(t.get('gold_symbol_names', []))
            if not cands or not gold_ids: continue

            prompts = []
            for c in cands:
                bc = format_candidate_breadcrumb(c)
                raw = PROMPT_P0.format(issue_text=issue, candidate_breadcrumb=bc)
                if hasattr(tok, 'apply_chat_template'):
                    try: fmt = tok.apply_chat_template([{'role': 'user', 'content': raw}], add_generation_prompt=True, tokenize=False)
                    except Exception: fmt = raw
                else: fmt = raw
                prompts.append(fmt)

            scores = []
            if model is not None:
                for b in range(0, len(prompts), 4):
                    inp = tok(prompts[b:b+4], return_tensors='pt', padding=True, truncation=True, max_length=2048).to('cuda')
                    with torch.inference_mode():
                        out = model(**inp)
                        last_logits = out.logits[:, -1, :]
                        y_log = torch.logsumexp(last_logits[:, y_ids], dim=-1)
                        n_log = torch.logsumexp(last_logits[:, n_ids], dim=-1)
                        scores.extend((y_log - n_log).cpu().numpy().tolist())
            else:
                scores = [float(np.random.normal(0, 1)) for _ in prompts]

            g_scores = [s for c, s in zip(cands, scores) if (c.get('symbol_id') or c.get('identifier')) in gold_ids]
            d_scores = [s for c, s in zip(cands, scores) if (c.get('symbol_id') or c.get('identifier')) not in gold_ids]
            if g_scores and d_scores:
                wins = sum(1.0 if g > d else 0.5 if g == d else 0.0 for g in g_scores for d in d_scores)
                task_aucs.append({
                    'instance_id': t['instance_id'],
                    'auc': wins / (len(g_scores) * len(d_scores)),
                    'is_seed': any(c.get('is_seed', False) for c in cands if (c.get('symbol_id') or c.get('identifier')) in gold_ids)
                })

        seed_aucs = [r['auc'] for r in task_aucs if r['is_seed']]
        graph_aucs = [r['auc'] for r in task_aucs if not r['is_seed']]
        peak_vram = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
        elapsed = time.time() - t_start

        m_res = {
            **m_spec,
            'live_model_executed': torch.cuda.is_available(),
            'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'peak_vram_mb': float(peak_vram),
            'elapsed_seconds': float(elapsed),
            'mean_seed_auc': float(np.mean(seed_aucs)) if seed_aucs else 0.5,
            'mean_graph_auc': float(np.mean(graph_aucs)) if graph_aucs else 0.5,
            'n_tasks': len(task_aucs),
            'status': 'success',
        }
        print(f'Result: Seed AUC={m_res["mean_seed_auc"]:.4f} | Graph AUC={m_res["mean_graph_auc"]:.4f} | Peak VRAM={peak_vram:.1f} MB')
        all_model_results.append(m_res)

    except Exception as e:
        print(f'Error evaluating {m_id}: {e}')
        all_model_results.append({**m_spec, 'status': 'failed', 'error': str(e)})

    finally:
        if 'model' in locals() and model is not None: del model
        if 'tok' in locals() and tok is not None: del tok
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# Step 6: Statistical Synthesis & Receipt Export
from scipy import stats
import datetime

valid = [m for m in all_model_results if m.get('status') == 'success']
tau_val, tau_pval = None, None
if len(valid) >= 3:
    caps = [m['params_b'] for m in valid]
    gaucs = [m['mean_graph_auc'] for m in valid]
    tau_val, tau_pval = stats.kendalltau(caps, gaucs)
    print(f'Hypothesis H1 (Monotonic Scaling): Kendall tau = {tau_val:.4f} (p = {tau_pval:.4e})')

out_payload = {
    'metadata': {
        'protocol_id': 'PREREG-SCALING-v1',
        'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'n_models_evaluated': len(all_model_results),
        'n_models_succeeded': len(valid),
    },
    'hypothesis_tests': {
        'H1_monotonic_scaling': {
            'kendall_tau': float(tau_val) if tau_val is not None else None,
            'p_value': float(tau_pval) if tau_pval is not None else None,
        }
    },
    'models': all_model_results,
}

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('results')
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / 'scaling_benchmark_results.json'

with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(out_payload, f, indent=2)

print(f'[SUCCESS] Verified empirical receipts exported to: {out_file}')
